# NorthWind PLUS business agent — Relate by Neo4j + Neo4j Aura + Claude

Ask business questions about the **live** NorthWind PLUS simulation. The agent looks up proven Cypher templates in its
**relate** knowledge store (the ontology plus the query library), runs read queries on **Neo4j Aura**, can make one
governed change (the L1/L2/L3 PO approval bands) — and **learns**: every question is kept in relate as an
`Interaction`, and new queries you approve become templates that later questions find.

**Before you start**
1. Load the graph: in the Aura Query console, run `model/NorthWind_PLUS_Graph_Data_Model_Import_v2.3.2.cypher`
   from line 95 (*"// Let's clear all Nodes..."*) to line 1171. This **wipes the database** and builds NorthWind PLUS v2.3.2.
   Optionally also run the Recommendation Engine (lines 1271-1334).
   *(Already on v2.3.1 with data you want to keep? Run `model/NorthWind_PLUS_v2.3.2_Upgrade_from_v2.3.1.cypher` instead.)*
2. Run `model/NorthwindPlusGraphType_v2.3.2.cypher` in the same console, to apply the GRAPH TYPE schema.
3. Start the six Python loops (`model/NorthwindPlus_Stress_Test.py`, one process per `--loop`), so the data is moving.
4. Fill in `.env` in this folder (see `.env.example`): `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.
5. Use a kernel where `pip install -r requirements.txt` has been run.

Run the cells from the top. Each question is one `ask(...)` call.

## 1. Setup

In [ ]:
import logging, os, sys, json, subprocess
from pathlib import Path

PROJECT = Path.cwd()                   # the notebook lives in the project folder
assert (PROJECT / "agent" / "agent.py").exists(), f"Open this notebook from the project folder (found {PROJECT})"
for sub in ("agent", "kb"):
    sys.path.insert(0, str(PROJECT / sub))
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

from IPython.display import Markdown, display
import agent as northwind_agent
from build_kb import build
from kb_query import KnowledgeBase
print("Project folder:", PROJECT)

## 2. Knowledge store (relate)

`EMBEDDINGS = "none"` uses full-text + graph + past-questions retrieval. `"voyage-4-nano"` adds semantic search with
Voyage AI's open-weight model, run on this machine (no API key; ~700 MB download the first time; needs
`sentence-transformers` and `torch`). Changing it rebuilds the store; **what the agent has learned is kept**.

In [ ]:
EMBEDDINGS = "none"        # or "voyage-4-nano"
REBUILD = False            # rebuild after editing kb/semantics.json or the model files (learning is kept)

store = PROJECT / "kb" / "northwind_kb.rl"
current = "none"
if store.exists():
    k = KnowledgeBase(store); current = k.embedding_model; k.close()
if REBUILD or not store.exists() or current != EMBEDDINGS:
    print(json.dumps(build(store, embeddings=EMBEDDINGS), indent=2))
else:
    print(f"Using {store.name} (embeddings: {current})")

## 3. Sign in

`EMPLOYEE_ID` stands in for a login — the session decides who you are, the model never does.

| EmployeeID | Who | Can change approval bands? |
|---|---|---|
| `25` | Olivia Bennett, ProcurementPolicyAdmin | yes |
| `12` | Gloria Gaynor, Level1Approver | no (can still ask questions) |

Rerunning this cell closes the previous session first (relate allows one writing connection per store).

In [ ]:
EMPLOYEE_ID = "25"
MODEL = northwind_agent.DEFAULT_MODEL      # e.g. "claude-opus-5-5" for harder questions

if "session" in globals():
    try:
        session.close()
    except Exception:
        pass
session = northwind_agent.Session(EMPLOYEE_ID, MODEL)
print(f"Signed in as {session.user['name']} ({', '.join(session.user['roles']) or 'no roles'}) -- model {session.model}")
print(f"Knowledge store: ontology {session.kb.ontology_version}, embeddings {session.kb.embedding_model}")


def ask(question: str):
    """Ask the agent. Tool calls print as they happen; the answer renders as Markdown."""
    display(Markdown(session.ask(question)))


def new_conversation():
    """Forget the conversation so far (same sign-in)."""
    session.messages.clear()
    print("Conversation cleared.")

## 4. Ask questions

The conversation carries over between cells, so follow-ups like *"and for last week?"* work. When the agent writes a
new query that worked, it offers to save it as a template: a text box appears under the cell — `y` to keep it.

In [ ]:
ask("Which suppliers get their quotes rejected most often?")

In [ ]:
ask("How many POs are waiting for approval?")

Ask the same thing in other words: if you saved the template above, the search should now find it — through
the template itself and through the past question it answered.

In [ ]:
ask("What is the number and value of purchase orders still waiting to be vetted?")

### Governed change

The agent previews first, then asks you to confirm in a text box under the cell: `y` to apply, anything else to cancel.
Only EmployeeID 25 can apply it.

In [ ]:
ask("Raise L1 to 8000 and L2 to 20000 to speed up vetting.")

In [ ]:
ask("Who changed the approval limits, when and why?")

### Your questions

In [ ]:
# ask("Which products are at risk of running out?")

## 5. What the agent learned (from relate)

Templates the agent created, how often templates get reused, and the question history — all read from the knowledge
store, the same graph the agent searches.

In [ ]:
kb = session.kb
learned = kb._rows("""
    MATCH (q:QueryTemplate {source:'agent'})
    OPTIONAL MATCH (q)-[:PART_OF]->(w:WorkflowStep)
    RETURN q.title AS title, q.createdBy AS by, q.question AS firstQuestion, coalesce(q.useCount, 0) AS reuses,
           q.status AS status, collect(DISTINCT w.name) AS workflows ORDER BY reuses DESC""")
display(Markdown("**Learned templates**\n\n| Title | First question | Created by | Reuses | Workflows |\n|---|---|---|---|---|\n" +
    "\n".join(f"| {r['title']} | {r['firstQuestion']} | {r['by']} | {r['reuses']} | {', '.join(r['workflows'])} |" for r in learned)
    if learned else "_No learned templates yet._"))

top = kb._rows("""MATCH (q:QueryTemplate) WHERE q.useCount > 0
                 RETURN q.title AS title, q.source AS source, q.useCount AS uses ORDER BY uses DESC LIMIT 10""")
display(Markdown("**Most used templates**\n\n| Template | Source | Uses |\n|---|---|---|\n" +
    "\n".join(f"| {r['title'][:90]} | {r['source']} | {r['uses']} |" for r in top) if top else "_No template used yet._"))

hist = kb._rows("""MATCH (i:Interaction) OPTIONAL MATCH (i)-[:USED|CREATED]->(q:QueryTemplate)
                  RETURN i.question AS q, i.employeeId AS who, i.modelCalls AS calls, i.inputTokens AS tin,
                         i.cacheReadTokens AS tcache, i.outputTokens AS tout, collect(DISTINCT q.title) AS templates,
                         i.newQueries AS newq, i.startedAt AS t ORDER BY t DESC LIMIT 15""")
display(Markdown("**Recent questions**\n\n| Question | By | Templates | New queries | Model calls | Tokens in (+cached) / out |\n|---|---|---|---|---|---|\n" +
    "\n".join(f"| {r['q']} | {r['who']} | {'<br>'.join(t[:60] for t in r['templates']) or '-'} | {r['newq'] or 0} | {r['calls']} | "
               f"{(r['tin'] or 0):,} (+{(r['tcache'] or 0):,}) / {(r['tout'] or 0):,} |" for r in hist)))

## 6. Done

In [ ]:
session.close()
print("Closed the Aura connection and the knowledge store. What the agent learned stays in kb/northwind_kb.rl.")